# 02 – Làm sạch nhẹ và ẩn danh hóa (Bronze → Silver)

**Mục đích:** đọc `data/raw/`, ghi `data/anonymized/`; là minh chứng cho mục 3.5 (và phần chuẩn hóa của mục 3.3).

**Nguyên tắc**
- Notebook này là nơi **duy nhất** ghi vào `data/anonymized/`. ETL (03, 04) chỉ đọc từ đây.
- Làm sạch ở tầng này **không xóa dòng**: các ngoại lệ chỉ được gắn cờ (`is_*`), ETL quyết định lọc. Nhờ vậy khóa ngoại không bị gãy và mọi quyết định lọc đều truy vết được.
- Muối băm lấy từ biến môi trường `ANON_SALT` (file `.env`), không ghi vào code, không ghi vào log.

**Các bước:** (1) kiểm tra muối → (2) bảng kê trường định danh → (3) ẩn danh ER → (4) ẩn danh Synthea → (5) kiểm tra k-anonymity → (6) quét sót định danh → (7) ghi nhật ký truy cập.

In [ ]:
from pathlib import Path
import os, re, json, hmac, hashlib, getpass
from datetime import datetime, timezone
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
ANON = ROOT / 'data' / 'anonymized'
PROC = ROOT / 'data' / 'processed'
ANON.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')
SALT = os.getenv('ANON_SALT')

K_MIN = 5                      # ngưỡng k-anonymity của nhóm
REF_DATE = pd.Timestamp('2021-12-31')   # mốc tính tuổi cho Synthea (dữ liệu kết thúc 11/2021)
AGE_BINS = [0, 18, 45, 65, 200]
AGE_LABELS = ['0-17', '18-44', '45-64', '65+']   # dùng chung cho cả hai mart

## 1. Kiểm tra muối
Dừng ngay nếu thiếu muối hoặc còn giá trị mẫu trong `.env.example`. Muối yếu làm phép băm có thể bị dò ngược (mã dạng SSN chỉ có khoảng 10^9 khả năng).

In [ ]:
assert SALT, "Thiếu ANON_SALT: sao chép .env.example thành .env và đặt muối thật."
assert SALT != 'doi_thanh_chuoi_ngau_nhien_dai', "ANON_SALT vẫn là giá trị mẫu."
assert len(SALT) >= 16, "ANON_SALT quá ngắn (cần tối thiểu 16 ký tự)."

def pseudonymize(value):
    """HMAC-SHA256 có khóa là muối; lấy 16 ký tự hex đầu (64 bit). Cùng một mã gốc luôn ra cùng một mã băm."""
    return hmac.new(SALT.encode(), str(value).encode(), hashlib.sha256).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

audit = {'inputs': {}, 'outputs': {}, 'dropped_columns': {}}

def read_input(name, **kw):
    path = RAW / name
    df = pd.read_csv(path, **kw)
    audit['inputs'][name] = {'sha256': sha256_file(path), 'bytes': path.stat().st_size, 'rows': len(df)}
    return df

def write_output(df, name):
    path = ANON / name
    df.to_csv(path, index=False)
    audit['outputs'][name] = {'sha256': sha256_file(path), 'bytes': path.stat().st_size, 'rows': len(df), 'columns': list(df.columns)}

## 2. Bảng kê trường định danh
Phân loại theo HIPAA Safe Harbor: **trực tiếp** (phải bỏ hoặc băm), **gián tiếp / quasi-identifier** (khái quát hóa), **không định danh** (giữ).

In [ ]:
inventory = pd.DataFrame([
 # nguồn, trường, loại, xử lý, lý do
 ('ER', 'Patient Id', 'Trực tiếp', 'Băm HMAC-SHA256 có muối → patient_hash', 'Có dạng số an sinh xã hội (100% dòng khớp ###-##-####)'),
 ('ER', 'Patient First Inital', 'Trực tiếp', 'Bỏ', 'Một phần tên'),
 ('ER', 'Patient Last Name', 'Trực tiếp', 'Bỏ', 'Họ tên'),
 ('ER', 'Patient Admission Date', 'Gián tiếp', 'Làm tròn xuống giờ → visit_datetime', 'Ngày giờ chính xác là mốc dễ liên kết; vẫn cần giờ cho heatmap'),
 ('ER', 'Patient Age', 'Gián tiếp', 'Nhóm tuổi 4 bậc → age_group', 'Tuổi chính xác kết hợp giới tính, chủng tộc làm tăng nguy cơ'),
 ('ER', 'Patient Gender', 'Gián tiếp', 'Chuẩn hóa M/F → gender', 'Nhãn không thống nhất (M, F, Male, Female)'),
 ('ER', 'Patient Race', 'Gián tiếp', 'Giữ → race', 'Cần cho phân tích công bằng; k được kiểm tra ở mục 5'),
 ('ER', 'Department Referral / Admission Flag / Satisfaction / Waittime', 'Không định danh', 'Giữ', 'Chỉ số vận hành'),
 ('ER', 'Patient Admission Flag.1', 'Trùng lặp', 'Bỏ', 'Cột lặp, giống hệt Patient Admission Flag'),
 ('Synthea', 'patients.Id, encounters.PATIENT', 'Trực tiếp', 'Băm HMAC-SHA256 có muối (cùng hàm)', 'Giữ được khóa nối giữa hai bảng'),
 ('Synthea', 'SSN, DRIVERS, PASSPORT', 'Trực tiếp', 'Bỏ', 'Số giấy tờ'),
 ('Synthea', 'PREFIX, FIRST, LAST, SUFFIX, MAIDEN', 'Trực tiếp', 'Bỏ', 'Họ tên'),
 ('Synthea', 'ADDRESS, CITY, COUNTY, ZIP, LAT, LON, BIRTHPLACE', 'Trực tiếp / gián tiếp', 'Bỏ', 'Địa chỉ và tọa độ (Safe Harbor yêu cầu bỏ mức nhỏ hơn bang)'),
 ('Synthea', 'BIRTHDATE', 'Gián tiếp', 'Bỏ; thay bằng age_group', 'Ngày sinh đầy đủ là định danh'),
 ('Synthea', 'DEATHDATE', 'Gián tiếp', 'Bỏ; thay bằng is_deceased', 'Ngày mất là định danh'),
 ('Synthea', 'STATE', 'Hằng số', 'Bỏ', 'Toàn bộ là Massachusetts'),
 ('Synthea', 'RACE, ETHNICITY, GENDER, MARITAL', 'Gián tiếp', 'Gộp nhóm hiếm, ẩn cục bộ nếu cần', 'Mẫu chỉ 1.163 người nên nhóm nhỏ dễ lộ'),
 ('Synthea', 'providers.NAME, ADDRESS, ZIP, LAT, LON', 'Trực tiếp', 'Bỏ', 'Tên và địa chỉ bác sĩ'),
 ('Synthea', 'organizations.ADDRESS, PHONE, ZIP, LAT, LON', 'Địa chỉ cơ sở', 'Bỏ', 'Không cần cho phân tích'),
 ('Synthea', 'organizations.REVENUE', 'Không dùng được', 'Bỏ', 'Toàn bằng 0'),
], columns=['source', 'field', 'category', 'treatment', 'reason'])
inventory.to_csv(ANON / 'identifier_inventory.csv', index=False)
inventory

## 3. ER → `er_visits_anon.csv`
Chú ý quan trọng: nhãn `None` ở `Department Referral` nghĩa là **không chuyển khoa**. Pandas mặc định coi `"None"` là thiếu dữ liệu, nên phải đọc với `keep_default_na=False` rồi chỉ coi chuỗi rỗng là thiếu.

In [ ]:
er = read_input('Hospital Emergency Room Data Raw.csv', keep_default_na=False, na_values=[''])
er.columns = [c.strip() for c in er.columns]

# Cột lặp: pandas tự đổi tên cột trùng thành 'Patient Admission Flag.1'
assert (er['Patient Admission Flag'] == er['Patient Admission Flag.1']).all()

dt = pd.to_datetime(er['Patient Admission Date'], format='%d-%m-%Y %H:%M')
gender = er['Patient Gender'].str.strip().str.upper().str[0]
assert gender.isin(['M', 'F']).all()

er_anon = pd.DataFrame({
    'patient_hash': er['Patient Id'].map(pseudonymize),
    'visit_datetime': dt.dt.floor('h').dt.strftime('%Y-%m-%d %H:%M'),
    'gender': gender,
    'age_group': pd.cut(er['Patient Age'], AGE_BINS, right=False, labels=AGE_LABELS).astype(str),
    'race': er['Patient Race'].str.strip(),
    'referral_department': er['Department Referral'].replace({'None': 'No referral'}),
    'is_referred': er['Department Referral'].ne('None'),
    'is_admitted': er['Patient Admission Flag'].astype(bool),
    'satisfaction_score': er['Patient Satisfaction Score'],       # giữ NaN, không điền (thiếu ~73%)
    'wait_time_min': er['Patient Waittime'],
})
assert er_anon['patient_hash'].is_unique                  # băm không va chạm
assert er_anon['age_group'].ne('nan').all()
audit['dropped_columns']['er'] = ['Patient Id', 'Patient First Inital', 'Patient Last Name', 'Patient Age', 'Patient Admission Date', 'Patient Admission Flag.1']
write_output(er_anon, 'er_visits_anon.csv')
er_anon.head()

## 4. Synthea → 4 file `*_anon.csv`
### 4.1 Bệnh nhân
Tuổi tính tại ngày mất (nếu đã mất) hoặc tại mốc 31/12/2021. Sau khi gộp nhóm, các nhóm tương đương còn nhỏ hơn k sẽ bị **ẩn cục bộ** (đổi giá trị thuộc tính thành `Suppressed`) thay vì xóa bệnh nhân, để không làm mất khóa ngoại.

In [ ]:
pat = read_input('patients.csv')
birth = pd.to_datetime(pat['BIRTHDATE'])
end = pd.to_datetime(pat['DEATHDATE']).fillna(REF_DATE).clip(upper=REF_DATE)
age = (end - birth).dt.days // 365.25

pat_anon = pd.DataFrame({
    'patient_hash': pat['Id'].map(pseudonymize),
    'gender': pat['GENDER'],
    'age_group': pd.cut(age, AGE_BINS, right=False, labels=AGE_LABELS).astype(str),
    'race': pat['RACE'].where(pat['RACE'].isin(['white', 'black', 'asian']), 'other'),   # gộp hawaiian, native, other
    'ethnicity': pat['ETHNICITY'],
    'marital_status': pat['MARITAL'].fillna('Unknown'),
    'is_deceased': pat['DEATHDATE'].notna(),
    'healthcare_expenses_usd': pat['HEALTHCARE_EXPENSES'],
    'healthcare_coverage_usd': pat['HEALTHCARE_COVERAGE'],
})
assert pat_anon['patient_hash'].is_unique

# Ẩn cục bộ theo thứ tự ưu tiên: thuộc tính ít giá trị phân tích nhất bị ẩn trước
QI_PAT = ['age_group', 'gender', 'race', 'ethnicity']

def class_sizes(df, cols):
    return df.groupby(cols)[cols[0]].transform('size')

suppression_log = []
for col in ['race', 'ethnicity', 'age_group']:
    small = class_sizes(pat_anon, QI_PAT) < K_MIN
    if not small.any():
        break
    pat_anon.loc[small, col] = 'Suppressed'
    suppression_log.append({'column': col, 'cells_suppressed': int(small.sum())})
suppression_log

`marital_status` không nằm trong bộ thuộc tính gián tiếp được bảo đảm k ≥ 5 (thêm vào sẽ phải ẩn quá nhiều ô); rủi ro này được nêu ở báo cáo mục 3.5.

In [ ]:
audit['dropped_columns']['patients'] = ['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON']
write_output(pat_anon, 'patients_anon.csv')

### 4.2 Lượt khám
Làm sạch **không xóa dòng**, chỉ gắn cờ để ETL lọc:
- `is_in_analysis_period`: bắt đầu từ năm 2011 (trước đó thưa, lệch dữ liệu).
- `is_cost_outlier`: chi phí vượt phân vị 99 (P99) của **riêng từng loại lượt khám**, tính trong giai đoạn phân tích và bỏ các dòng chi phí 0.
  (Đã thử IQR trên log(1 + chi phí) nhưng bỏ: ngưỡng trên lên tới hàng triệu USD nên không bắt được ca cực đoan nào, còn phía dưới lại gắn cờ nhầm các lượt wellness giá rẻ hợp lệ.)
- `is_zero_cost`: tổng chi phí bằng 0.
- `is_duration_outlier`: thời lượng > 24 giờ với loại không phải nội trú; nội trú > 30 ngày.
- `reason_description`: trống được điền `Not recorded` và có cờ `has_reason`.

In [ ]:
enc = read_input('encounters.csv', low_memory=False)
start = pd.to_datetime(enc['START'], utc=True)
stop = pd.to_datetime(enc['STOP'], utc=True)
duration_min = (stop - start).dt.total_seconds() / 60

in_period = start.dt.year >= 2011
cost = enc['TOTAL_CLAIM_COST']
cost_outlier = pd.Series(False, index=enc.index)
for cls, idx in enc.groupby('ENCOUNTERCLASS').groups.items():
    ref = cost.loc[[i for i in idx if in_period.loc[i] and cost.loc[i] > 0]]
    p99 = ref.quantile(0.99)
    cost_outlier.loc[idx] = cost.loc[idx] > p99

dur_limit = np.where(enc['ENCOUNTERCLASS'].eq('inpatient'), 30 * 1440, 1440)

enc_anon = pd.DataFrame({
    'encounter_id': enc['Id'],
    'patient_hash': enc['PATIENT'].map(pseudonymize),
    'organization_id': enc['ORGANIZATION'],
    'provider_id': enc['PROVIDER'],
    'payer_id': enc['PAYER'],
    'encounter_class': enc['ENCOUNTERCLASS'],
    'encounter_code': enc['CODE'],
    'encounter_description': enc['DESCRIPTION'],
    'start_datetime': start.dt.strftime('%Y-%m-%d %H:%M:%S'),
    'stop_datetime': stop.dt.strftime('%Y-%m-%d %H:%M:%S'),
    'duration_min': duration_min.round(1),
    'base_cost_usd': enc['BASE_ENCOUNTER_COST'],
    'total_cost_usd': enc['TOTAL_CLAIM_COST'],
    'payer_coverage_usd': enc['PAYER_COVERAGE'],
    'reason_description': enc['REASONDESCRIPTION'].fillna('Not recorded'),
    'has_reason': enc['REASONDESCRIPTION'].notna(),
    'is_in_analysis_period': in_period,
    'is_cost_outlier': cost_outlier,
    'is_zero_cost': enc['TOTAL_CLAIM_COST'].eq(0),
    'is_duration_outlier': duration_min > dur_limit,
})
# Kiểm tra: mọi khóa nối vẫn khớp sau khi băm
assert enc_anon['patient_hash'].isin(pat_anon['patient_hash']).all()
audit['dropped_columns']['encounters'] = ['Id (đổi tên)', 'PATIENT (băm)', 'REASONCODE']
write_output(enc_anon, 'encounters_anon.csv')

enc_anon[['is_in_analysis_period', 'is_cost_outlier', 'is_zero_cost', 'is_duration_outlier', 'has_reason']].sum()

### 4.3 Nhà cung cấp và tổ chức

In [ ]:
prov = read_input('providers.csv')
prov_anon = pd.DataFrame({
    'provider_id': prov['Id'],
    'organization_id': prov['ORGANIZATION'],
    'gender': prov['GENDER'],
    'specialty': prov['SPECIALITY'],
    'city': prov['CITY'],
    'utilization': prov['UTILIZATION'],
})
audit['dropped_columns']['providers'] = ['NAME', 'ADDRESS', 'STATE', 'ZIP', 'LAT', 'LON']
write_output(prov_anon, 'providers_anon.csv')

org = read_input('organizations.csv')
assert (org['REVENUE'] == 0).all()     # xác nhận REVENUE vô dụng trước khi bỏ
org_anon = pd.DataFrame({
    'organization_id': org['Id'],
    'organization_name': org['NAME'].str.replace(r'\s+', ' ', regex=True).str.strip(),
    'city': org['CITY'],
    'utilization': org['UTILIZATION'],
})
audit['dropped_columns']['organizations'] = ['ADDRESS', 'STATE', 'ZIP', 'LAT', 'LON', 'PHONE', 'REVENUE']
write_output(org_anon, 'organizations_anon.csv')

## 5. Kiểm tra k-anonymity
k = kích thước nhóm tương đương nhỏ nhất theo bộ thuộc tính gián tiếp. Yêu cầu của nhóm: **k ≥ 5** với bộ thuộc tính nhân khẩu học. Các bộ có thêm thời gian được đo và công bố nhưng **không** ràng buộc, vì phân tích vận hành cần giờ vào (xem rủi ro còn lại ở báo cáo).

In [ ]:
def k_report(df, cols, dataset, label):
    sizes = df.groupby(cols).size()
    per_row = df.groupby(cols)[cols[0]].transform('size')
    return {
        'dataset': dataset, 'qi_set': label, 'qi_columns': '+'.join(cols),
        'k_min': int(sizes.min()), 'equivalence_classes': int(len(sizes)),
        'records_in_class_lt_k': int((per_row < K_MIN).sum()),
        'pct_records_in_class_lt_k': round(100 * (per_row < K_MIN).mean(), 1),
        'unique_records': int((per_row == 1).sum()),
    }

er_k = er_anon.assign(visit_month=er_anon['visit_datetime'].str[:7], visit_hour=er_anon['visit_datetime'].str[11:13])
rows = [
    k_report(er_k, ['age_group', 'gender', 'race'], 'ER', 'A: nhân khẩu học (ràng buộc)'),
    k_report(er_k, ['age_group', 'gender', 'race', 'referral_department'], 'ER', 'B: A + khoa chuyển tuyến'),
    k_report(er_k, ['age_group', 'gender', 'race', 'visit_month'], 'ER', 'C: A + tháng khám'),
    k_report(er_k, ['age_group', 'gender', 'race', 'visit_hour'], 'ER', 'D: A + giờ khám'),
    k_report(pat_anon, QI_PAT, 'Synthea', 'A: nhân khẩu học sau ẩn cục bộ (ràng buộc)'),
    k_report(pat_anon.assign(m=pat_anon['marital_status']), QI_PAT + ['m'], 'Synthea', 'B: A + tình trạng hôn nhân'),
]
k_table = pd.DataFrame(rows)
k_table.to_csv(ANON / 'k_anonymity_report.csv', index=False)
audit['k_anonymity'] = rows
audit['suppression'] = suppression_log

# Ràng buộc cứng: bộ A của hai nguồn phải đạt k ≥ 5
assert k_table.loc[k_table['qi_set'].str.startswith('A'), 'k_min'].min() >= K_MIN, "k-anonymity chưa đạt"
k_table.drop(columns='qi_columns')

## 6. Quét sót định danh trong đầu ra
Tìm mẫu số an sinh xã hội trong mọi cột chữ, và kiểm tra không còn mã gốc nào lọt sang tầng Silver.

In [ ]:
ssn_pattern = re.compile(r'\b\d{3}-\d{2}-\d{4}\b')
problems = []
outputs = {'er_visits_anon.csv': er_anon, 'patients_anon.csv': pat_anon, 'encounters_anon.csv': enc_anon,
           'providers_anon.csv': prov_anon, 'organizations_anon.csv': org_anon}
for name, df in outputs.items():
    for col in df.select_dtypes(include=['object', 'string']).columns:
        n = df[col].astype(str).str.contains(ssn_pattern).sum()
        if n:
            problems.append((name, col, int(n)))

raw_ids = set(er['Patient Id']) | set(pat['Id'])
leak_er = er_anon['patient_hash'].isin(raw_ids).sum()
leak_pat = pat_anon['patient_hash'].isin(raw_ids).sum()
forbidden_cols = {'Patient Id', 'Patient Last Name', 'SSN', 'FIRST', 'LAST', 'ADDRESS', 'LAT', 'LON', 'BIRTHDATE', 'NAME'}
leftover = [(n, c) for n, d in outputs.items() for c in d.columns if c in forbidden_cols]

assert not problems, f"Còn mẫu SSN: {problems}"
assert leak_er == 0 and leak_pat == 0
assert not leftover, f"Còn cột định danh: {leftover}"
print('Quét sót: không còn mẫu SSN, mã gốc hay cột định danh trong 5 file đầu ra.')

## 7. Nhật ký truy cập (audit log)
Mỗi lần chạy thêm một dòng vào `anonymization_audit.jsonl`: thời điểm UTC, người chạy, hash SHA-256 và số dòng của file vào/ra, cột đã bỏ, kết quả k-anonymity. Log **không** chứa muối hay dữ liệu bệnh nhân.
Nhật ký này chỉ ghi các lần chạy notebook; nhật ký truy cập dữ liệu ở mức hệ thống (ai mở file, ai truy vấn kho) phải do hệ quản trị CSDL hoặc kho lưu trữ ghi, được mô tả ở báo cáo mục 3.5.

In [ ]:
audit_record = {'run_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'run_by': getpass.getuser(),
                'salt_configured': True, 'salt_length': len(SALT), 'k_threshold': K_MIN, **audit}
with open(ANON / 'anonymization_audit.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(audit_record, ensure_ascii=False) + '\n')
print('Đã ghi:', *sorted(p.name for p in ANON.glob('*') if p.name != '.gitkeep'), sep='\n- ')